NLSY79(1979-2020) is the source of the data that I obtain from https://www.nlsinfo.org/. I begin by downloading from the employment history, then I proceed to go through the week arrays, and finally I download all of the employment statuses for the weeks. 

In [ ]:
using DataFrames, CSV, Dates, Plots, DataFramesMeta
using Douglass
using RDatasets
using RCall

In [ ]:
# Load the dataset D:\PSID_SHELF\02_NLSY79\data\NLSY79.CSV
df = CSV.read("D:\\PSID_SHELF\\02_NLSY79\\Data\\NLSY79.CSV", DataFrame)

In [ ]:
# Get the names of the columns
col_names = names(df)

# Iterate over each column name
for col in col_names
    # Check if the column name contains "_XRND" or "NUM"
    if occursin("_XRND", col) || occursin("NUM", col)
        # Remove the "_XRND" substring
        new_col = replace(col, "_XRND" => "") 
        # Remove the "NUM" substring
        new_col = replace(new_col, "NUM" => "")
        # Rename the column
        DataFrames.rename!(df, col => new_col)
    end
end

# Get the names of the columns after renaming
col_names = names(df)

In [ ]:
#RENAME SAMPWEIGHT TO SAMPWEIGHT_0
DataFrames.rename!(df, :SAMPWEIGHT => :SAMPWEIGHT_0)

In [ ]:
mapping = Dict(
    0 => 1979, 1 => 1980, 2 => 1981, 3 => 1982, 4 => 1983, 5 => 1984, 6 => 1985, 7 => 1986, 8 => 1987, 9 => 1988, 
    10 => 1989, 11 => 1990, 12 => 1991, 13 => 1992, 14 => 1993, 15 => 1994, 16 => 1996, 17 => 1998, 18 => 2000, 
    19 => 2002, 20 => 2004, 21 => 2006, 22 => 2008, 23 => 2010, 24 => 2012, 25 => 2014, 26 => 2016, 27 => 2018, 28 => 2020
)
#RENAME SAMPWEIGHT_0 to SAMPWEIGHT_28 by mapping Dict 
for i in 0:28
    DataFrames.rename!(df, Symbol("SAMPWEIGHT_$(i)") => Symbol("SAMPWEIGHT_$(mapping[i])"))
end

# write column TO data frame EXAVCTLY as same as  SAMPWEIGHT_1996 and name it SAMPWEIGHT_1995 , do it again for SAMPWEIGHT_1998 and name it SAMPWEIGHT_1997 , do it again for SAMPWEIGHT_2000 and name it SAMPWEIGHT_1999, do it again for SAMPWEIGHT_2002 and name it SAMPWEIGHT_2001, do it again for SAMPWEIGHT_2004 and name it SAMPWEIGHT_2003, do it again for SAMPWEIGHT_2006 and name it SAMPWEIGHT_2005, do it again for SAMPWEIGHT_2008 and name it SAMPWEIGHT_2007, do it again for SAMPWEIGHT_2010 and name it SAMPWEIGHT_2009, do it again for SAMPWEIGHT_2012 and name it SAMPWEIGHT_2011, do it again for SAMPWEIGHT_2014 and name it SAMPWEIGHT_2013, do it again for SAMPWEIGHT_2016 and name it SAMPWEIGHT_2015, do it again for SAMPWEIGHT_2018 and name it SAMPWEIGHT_2017, do it again for SAMPWEIGHT_2020 and name it SAMPWEIGHT_2019
df[! ,:SAMPWEIGHT_1995] = df[! ,:SAMPWEIGHT_1996]
df[! ,:SAMPWEIGHT_1997] = df[! ,:SAMPWEIGHT_1998]
df[! ,:SAMPWEIGHT_1999] = df[! ,:SAMPWEIGHT_2000]
df[! ,:SAMPWEIGHT_2001] = df[! ,:SAMPWEIGHT_2002]
df[! ,:SAMPWEIGHT_2003] = df[! ,:SAMPWEIGHT_2004]
df[! ,:SAMPWEIGHT_2005] = df[! ,:SAMPWEIGHT_2006]
df[! ,:SAMPWEIGHT_2007] = df[! ,:SAMPWEIGHT_2008]
df[! ,:SAMPWEIGHT_2009] = df[! ,:SAMPWEIGHT_2010]
df[! ,:SAMPWEIGHT_2011] = df[! ,:SAMPWEIGHT_2012]
df[! ,:SAMPWEIGHT_2013] = df[! ,:SAMPWEIGHT_2014]
df[! ,:SAMPWEIGHT_2015] = df[! ,:SAMPWEIGHT_2016]
df[! ,:SAMPWEIGHT_2017] = df[! ,:SAMPWEIGHT_2018]
df[! ,:SAMPWEIGHT_2019] = df[! ,:SAMPWEIGHT_2020]

#write two column TO data frame EXAVCTLY as same as  SAMPWEIGHT_1979 and name it SAMPWEIGHT_1978 and SAMPWEIGHT_1977
df[! ,:SAMPWEIGHT_1978] = df[! ,:SAMPWEIGHT_1979]
df[! ,:SAMPWEIGHT_1977] = df[! ,:SAMPWEIGHT_1979]



In [ ]:
stacked_df = stack(df, Between(:STATUS_WK_0000, :STATUS_WK_2296), [:CASEID], variable_name=:week, value_name=:status)

In [ ]:
#STATUS_WK_0000 = 1 TO STATUS_WK_2296 = 2296 , RENAME value_name
df2 = @transform(stacked_df, :week = parse.(Int, replace.(:week, "STATUS_WK_" => "")))


In [ ]:
df3 = select(df, Not(Between(:STATUS_WK_0000, :STATUS_WK_2296)))

In [ ]:
# df_long = df2 + df_stacked on CASEID
df = leftjoin(df3, df2, on=:CASEID)

In [ ]:
# Set the starting date for all entries
df[!, :date] = [Date(1977, 12, 25) + Day(7 * (wk - 1)) for wk in df[!, :week]]
df[!, :date] = Date.(df[!, :date])

In [ ]:
# Extract year and month for each observation
df[!, :year] = year.(df[!, :date])
df[!, :month] = month.(df[!, :date])
df[!, :WEEK] = week.(df[!, :date])

In [ ]:
# Sort data by household ID and date
sort!(df, [:CASEID, :date])

In [ ]:
# write coulmn weight zero that has row as length as of CASEID
df[! ,:weight] = zeros(size(df, 1))

# if year of date = i of SAMWEIGHT_i then ofr all rows that years of date = i , weight = SAMPWEIGHT_i
for i in 1977:2020
    df[! ,:weight] = ifelse.(df[! ,:year] .== i, df[! ,Symbol("SAMPWEIGHT_$(i)")], df[! ,:weight])
end


In [ ]:
df = select!(df, Not([Symbol("SAMPWEIGHT_$(i)") for i in 1977:2020]))

In [ ]:
@rput df

In [ ]:
using Pkg
Pkg.update()


In [ ]:
CSV.write("D:/PSID_SHELF/02_NLSY79/data/Long_Employment_NLSY79.csv", df)
